In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, WhisperProcessor, WhisperForConditionalGeneration
import librosa
import numpy as np
import re
from typing import List, Dict, Tuple
import json
import soundfile as sf
import tempfile
import os
from difflib import SequenceMatcher
from scipy.signal import find_peaks

class ImprovedKoreanPronunciationAssessor:
    def __init__(self):
        print("Loading models...")
        # Load Whisper model (using larger model for better accuracy)
        self.whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
        self.whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")

        # Load Korean Wav2Vec2 for additional validation
        self.wav2vec_processor = Wav2Vec2Processor.from_pretrained("kresnik/wav2vec2-large-xlsr-korean")
        self.wav2vec_model = Wav2Vec2ForCTC.from_pretrained("kresnik/wav2vec2-large-xlsr-korean")

        # Move to GPU if available
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.whisper_model.to(device)
        self.wav2vec_model.to(device)
        self.device = device
        print(f"Models loaded on {device}")

    def get_full_sentence_transcription(self, audio_path: str) -> Tuple[str, float]:
        """Get full sentence transcription with confidence using both models"""

        # Load audio
        audio, sr = librosa.load(audio_path, sr=16000)

        # Get Whisper transcription - Fixed input handling
        whisper_text = ""
        whisper_confidence = 0.5

        try:
            # Process audio for Whisper
            whisper_inputs = self.whisper_processor(
                audio,
                sampling_rate=16000,
                return_tensors="pt"
            )

            # Handle different input feature names
            input_features = None
            for key in ['input_features', 'input_values']:
                if key in whisper_inputs:
                    input_features = whisper_inputs[key]
                    break

            if input_features is None:
                # Try to get the first tensor value
                for key, value in whisper_inputs.items():
                    if isinstance(value, torch.Tensor):
                        input_features = value
                        break

            if input_features is None:
                raise ValueError("Could not find input features in Whisper processor output")

            input_features = input_features.to(self.device)

            # Get language token for Korean
            forced_decoder_ids = self.whisper_processor.get_decoder_prompt_ids(
                language="korean",
                task="transcribe"
            )

            with torch.no_grad():
                whisper_outputs = self.whisper_model.generate(
                    input_features,
                    forced_decoder_ids=forced_decoder_ids,
                    max_length=200,
                    num_beams=5,
                    temperature=0.0,
                    do_sample=False
                )

            whisper_text = self.whisper_processor.batch_decode(
                whisper_outputs,
                skip_special_tokens=True
            )[0].strip()

            whisper_confidence = 0.9  # Default high confidence for sentence-level

        except Exception as e:
            print(f"Whisper processing error: {e}")
            whisper_text = ""
            whisper_confidence = 0.0

        # Get Wav2Vec2 transcription for validation
        wav2vec_text = ""

        try:
            wav2vec_inputs = self.wav2vec_processor(
                audio,
                sampling_rate=16000,
                return_tensors="pt",
                padding=True
            )

            # Fixed input handling for Wav2Vec2
            input_values = None
            for key in ['input_values', 'input_features']:
                if key in wav2vec_inputs:
                    input_values = wav2vec_inputs[key]
                    break

            if input_values is None:
                # Get first tensor value
                for key, value in wav2vec_inputs.items():
                    if isinstance(value, torch.Tensor):
                        input_values = value
                        break

            if input_values is not None:
                input_values = input_values.to(self.device)

                with torch.no_grad():
                    wav2vec_logits = self.wav2vec_model(input_values).logits
                    wav2vec_ids = torch.argmax(wav2vec_logits, dim=-1)
                    wav2vec_text = self.wav2vec_processor.batch_decode(wav2vec_ids)[0]

                # Clean up wav2vec text
                wav2vec_text = re.sub(r'\s+', ' ', wav2vec_text).strip()

        except Exception as e:
            print(f"Wav2Vec2 processing error: {e}")
            wav2vec_text = ""

        print(f"Whisper: {whisper_text}")
        print(f"Wav2Vec2: {wav2vec_text}")

        # Use the better transcription
        if whisper_text and len(whisper_text) > len(wav2vec_text):
            return whisper_text, whisper_confidence
        elif wav2vec_text:
            return wav2vec_text, 0.8  # Default confidence for wav2vec
        else:
            return whisper_text, whisper_confidence

    def calculate_levenshtein_similarity(self, str1: str, str2: str) -> float:
        """Calculate similarity using Levenshtein distance"""
        if not str1 or not str2:
            return 0.0

        # Calculate Levenshtein distance
        def levenshtein_distance(s1, s2):
            if len(s1) < len(s2):
                return levenshtein_distance(s2, s1)

            if len(s2) == 0:
                return len(s1)

            previous_row = list(range(len(s2) + 1))
            for i, c1 in enumerate(s1):
                current_row = [i + 1]
                for j, c2 in enumerate(s2):
                    insertions = previous_row[j + 1] + 1
                    deletions = current_row[j] + 1
                    substitutions = previous_row[j] + (c1 != c2)
                    current_row.append(min(insertions, deletions, substitutions))
                previous_row = current_row

            return previous_row[-1]

        distance = levenshtein_distance(str1, str2)
        max_len = max(len(str1), len(str2))
        similarity = 1 - (distance / max_len) if max_len > 0 else 0

        return max(0, similarity)

    def calculate_word_score_and_status(self, expected: str, transcribed: str, confidence: float, similarity: float) -> Tuple[str, int]:
        """Calculate word score and status"""

        expected_clean = re.sub(r'[^\w가-힣]', '', expected).strip()
        transcribed_clean = re.sub(r'[^\w가-힣]', '', transcribed).strip()

        # Base score calculation
        if not transcribed_clean:
            return 'incorrect', 15

        if expected_clean == transcribed_clean:
            # Perfect match
            if confidence > 0.9:
                return 'correct', 95
            elif confidence > 0.8:
                return 'correct', 90
            elif confidence > 0.7:
                return 'correct', 85
            else:
                return 'almost_correct', 80
        else:
            # Partial match - use similarity and confidence
            base_score = similarity * 100
            confidence_bonus = confidence * 10

            final_score = min(100, base_score + confidence_bonus)

            if final_score >= 85:
                return 'correct', int(final_score)
            elif final_score >= 70:
                return 'almost_correct', int(final_score)
            elif final_score >= 50:
                return 'almost_correct', int(final_score)
            else:
                return 'incorrect', int(final_score)

    def preprocess_compound_words(self, expected_words: List[str], transcribed_words: List[str]) -> List[str]:
        """Handle compound words that might be split during transcription"""

        # Try to reconstruct compound words from transcribed text
        reconstructed_transcribed = []
        i = 0

        for expected_word in expected_words:
            if i >= len(transcribed_words):
                reconstructed_transcribed.append("")
                continue

            # Check if this expected word might be split in transcription
            if len(expected_word) > 3:  # Likely compound word
                # Try to combine next few transcribed words
                combined = ""
                original_i = i
                max_combine = min(3, len(transcribed_words) - i)  # Try up to 3 words

                for j in range(max_combine):
                    if i + j < len(transcribed_words):
                        test_combined = combined + transcribed_words[i + j]
                        similarity = self.calculate_levenshtein_similarity(expected_word, test_combined)

                        if similarity > 0.7:  # Good match found
                            combined = test_combined
                            i += j + 1
                            break
                        elif j == 0:  # First word, keep it anyway
                            combined = test_combined
                else:
                    # No good match found, use first word only
                    i = original_i + 1

                reconstructed_transcribed.append(combined)
            else:
                reconstructed_transcribed.append(transcribed_words[i] if i < len(transcribed_words) else "")
                i += 1

        return reconstructed_transcribed

    def align_words_with_transcription(self, expected_words: List[str], transcribed_words: List[str], confidence: float) -> List[Dict]:
        """Align expected words with transcribed words and return simplified format"""

        # Handle compound words
        processed_transcribed = self.preprocess_compound_words(expected_words, transcribed_words)

        word_results = []

        # Align words one-to-one
        for i, expected_word in enumerate(expected_words):
            transcribed_word = processed_transcribed[i] if i < len(processed_transcribed) else ""

            # Calculate similarity
            similarity = self.calculate_levenshtein_similarity(expected_word, transcribed_word)

            # Get status and score
            status, score = self.calculate_word_score_and_status(expected_word, transcribed_word, confidence, similarity)

            word_results.append({
                "word": expected_word,
                "status": status
            })

        return word_results

    def create_highlighted_text(self, word_results: List[Dict]) -> str:
        """Create highlighted text by grouping consecutive words with same status"""

        if not word_results:
            return ""

        highlighted_parts = []
        current_group = []
        current_status = word_results[0]['status']

        for word_data in word_results:
            word = word_data['word']
            status = word_data['status']

            if status == current_status:
                current_group.append(word)
            else:
                # Save current group
                if current_group:
                    text = ' '.join(current_group)
                    highlighted_parts.append(f"<span class='{current_status}'>{text}</span>")

                # Start new group
                current_group = [word]
                current_status = status

        # Add the last group
        if current_group:
            text = ' '.join(current_group)
            highlighted_parts.append(f"<span class='{current_status}'>{text}</span>")

        return ' '.join(highlighted_parts)

    def generate_specific_feedback(self, expected_words: List[str], transcribed_words: List[str], word_results: List[Dict]) -> List[str]:
        """Generate specific pronunciation feedback"""

        feedback = []

        # Process compound words first
        processed_transcribed = self.preprocess_compound_words(expected_words, transcribed_words)

        for i, word_data in enumerate(word_results):
            expected = word_data['word']
            status = word_data['status']
            transcribed = processed_transcribed[i] if i < len(processed_transcribed) else ""

            if status == 'incorrect':
                if not transcribed:
                    feedback.append(f"'{expected}' 단어가 명확하게 들리지 않았어요. 더 크고 분명하게 발음해보세요.")
                else:
                    feedback.append(f"'{expected}'이 '{transcribed}'로 들렸어요. 정확한 발음을 연습해보세요.")

            elif status == 'almost_correct':
                # Analyze specific pronunciation issues
                if len(expected) > 0 and len(transcribed) > 0:
                    # Check for common Korean pronunciation issues
                    if '받침' in expected or any(char in expected for char in ['ㄱ', 'ㄴ', 'ㄷ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅅ']):
                        feedback.append(f"'{expected}'의 받침 발음이 약하게 들렸어요. 더 명확하게 발음해보세요.")
                    elif expected != transcribed:
                        feedback.append(f"'{expected}'이 거의 정확하지만 '{transcribed}'로 들렸어요. 조금만 더 정확하게 발음해보세요.")

        # If no specific feedback, give general encouragement
        if not feedback:
            correct_count = sum(1 for w in word_results if w['status'] == 'correct')
            if correct_count == len(word_results):
                feedback.append("완벽한 발음입니다! 👏")
            else:
                feedback.append("전반적으로 좋은 발음이에요. 계속 연습하세요!")

        return feedback[:3]  # Limit to 3 feedback items

    def calculate_pronunciation_score(self, word_results: List[Dict]) -> int:
        """Calculate overall pronunciation score"""

        if not word_results:
            return 0

        score_mapping = {
            'correct': 100,
            'almost_correct': 75,
            'incorrect': 25
        }

        total_score = sum(score_mapping.get(word['status'], 25) for word in word_results)
        average_score = total_score / len(word_results)

        return int(average_score)

    def assess_sentence_pronunciation(self, audio_path: str, text: str) -> Dict:
        """Main assessment function returning simplified format"""
        try:
            print("Starting pronunciation assessment...")

            # Step 1: Get full sentence transcription
            print("Getting full sentence transcription...")
            full_transcription, full_confidence = self.get_full_sentence_transcription(audio_path)

            # Step 2: Clean and prepare text for comparison
            expected_clean = re.sub(r'[^\w가-힣\s]', '', text).strip()
            transcribed_clean = re.sub(r'[^\w가-힣\s]', '', full_transcription).strip()

            expected_words = expected_clean.split()
            transcribed_words = transcribed_clean.split()

            print(f"Expected words: {expected_words}")
            print(f"Transcribed words: {transcribed_words}")

            # Step 3: Align words and get simplified results
            print("Aligning words with transcription...")
            word_results = self.align_words_with_transcription(expected_words, transcribed_words, full_confidence)

            # Step 4: Calculate overall score
            pronunciation_score = self.calculate_pronunciation_score(word_results)

            # Step 5: Create highlighted text
            highlighted_text = self.create_highlighted_text(word_results)

            # Step 6: Generate specific feedback
            processed_transcribed = self.preprocess_compound_words(expected_words, transcribed_words)
            feedback = self.generate_specific_feedback(expected_words, processed_transcribed, word_results)

            print("Assessment completed!")

            # Print results for debugging
            for word in word_results:
                print(f"Word '{word['word']}': {word['status']}")

            # Return in the requested format
            return {
                "sentence": text,
                "words": word_results,
                "pronunciation_score": pronunciation_score,
                "highlighted_text": highlighted_text,
                "feedback": feedback
            }

        except Exception as e:
            print(f"Error in assessment: {e}")
            import traceback
            traceback.print_exc()

            words = text.split()
            return {
                "sentence": text,
                "words": [{"word": word, "status": "incorrect"} for word in words],
                "pronunciation_score": 30,
                "highlighted_text": f"<span class='incorrect'>{text}</span>",
                "feedback": [f"처리 중 오류가 발생했습니다: {str(e)}"]
            }


def main():
    """Example usage"""
    assessor = ImprovedKoreanPronunciationAssessor()

    # Example usage
    audio_path = "T0307G0151S0018.wav"
    text = "김화영이 번역하고 책세상에서 출간된 카뮈의 전집"

    try:
        result = assessor.assess_sentence_pronunciation(audio_path, text)

        print("\n" + "="*50)
        print("KOREAN PRONUNCIATION ASSESSMENT RESULTS")
        print("="*50)
        print(json.dumps(result, ensure_ascii=False, indent=2))

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()

C:\Users\Tanuj\AppData\Roaming\Python\Python39\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading models...
Models loaded on cuda
Starting pronunciation assessment...
Getting full sentence transcription...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
C:\Users\Tanuj\AppData\Roaming\Python\Python39\site-packages\transformers\models\whisper\modeling_whisper.py:542: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Whisper: 김화영이 번역하고 책 세상에서 출간된 카뮤의 전집.
Wav2Vec2: 김화형이 번역하고 택 시상에서 출간된 카유에 전집
Expected words: ['김화영이', '번역하고', '책세상에서', '출간된', '카뮈의', '전집']
Transcribed words: ['김화영이', '번역하고', '책', '세상에서', '출간된', '카뮤의', '전집']
Aligning words with transcription...
Assessment completed!
Word '김화영이': correct
Word '번역하고': correct
Word '책세상에서': correct
Word '출간된': correct
Word '카뮈의': almost_correct
Word '전집': correct

KOREAN PRONUNCIATION ASSESSMENT RESULTS
{
  "sentence": "김화영이 번역하고 책세상에서 출간된 카뮈의 전집",
  "words": [
    {
      "word": "김화영이",
      "status": "correct"
    },
    {
      "word": "번역하고",
      "status": "correct"
    },
    {
      "word": "책세상에서",
      "status": "correct"
    },
    {
      "word": "출간된",
      "status": "correct"
    },
    {
      "word": "카뮈의",
      "status": "almost_correct"
    },
    {
      "word": "전집",
      "status": "correct"
    }
  ],
  "pronunciation_score": 95,
  "highlighted_text": "<span class='correct'>김화영이 번역하고 책세상에서 출간된</span> <span class='almost_correct'>